# 00 — Admission-level data audit

**Purpose.** Check the one-row-per-admission input before cohort splitting or model training. This notebook reports problems; it does not silently remove rows or alter predictor values.

**Important assumptions**
- The input contains one row per admission.
- The configured target is the outcome for supervised modelling.
- Missing or `unknown` outcomes remain visible in the audit and are excluded later during model fitting.
- Re-run this notebook after changing the input path or column definitions.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
print("Current Working Directory:", os.getcwd())

Current Working Directory: /app


In [3]:
from pathlib import Path
import yaml
import numpy as np

from icare_risk.data.audit import (
    load_admission_file, dataset_overview, 
    column_summary, duplicate_summary, 
    outcome_summary, temporal_summary, 
    save_audit_report,
)
from icare_risk.data.validation import (
    validate_admission_table, 
    validate_binary_target
)

from icare_risk.utils.paths import ANALYSIS_YAML_PATH
with ANALYSIS_YAML_PATH.open() as f:
    config = yaml.safe_load(f)

input_cfg = config['input']
columns = config['columns']
target_mapping = config.get('target_mapping', {})

print('Configuration:')
print(f"Input: {input_cfg['path']}")
print(f"Admission ID: {columns['admission_id']}")
print(f"Patient ID: {columns['patient_id']}")
print(f"Admission date: {columns['admission_date']}")
print(f"Target: {columns['target']}")
print(f"Positive labels: {target_mapping.get('positive_values', [])}")
print(f"Negative labels: {target_mapping.get('negative_values', [])}")

Configuration:
Input: /app/data/synthetic/10000p/processed/03_derived_analysis_dataset.parquet
Admission ID: SPELL_IDENTIFIER
Patient ID: SUBJECT
Admission date: ADMISSION_DATE
Target: esbl_label_test
Positive labels: ['esbl', 'positive', 1, '1']
Negative labels: ['non-esbl', 'non esbl', 'negative', 0, '0']


In [4]:
df = load_admission_file(
    path=input_cfg['path'],
    **input_cfg.get('read_csv_kwargs', {})
)
print(f"Loaded {len(df):,} rows and {df.shape[1]:,} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
display(df.head())

Loaded 10,000 rows and 189 columns
Memory usage: 21.83 MB


,SUBJECT,SPELL_IDENTIFIER,ENCNTR_ID,EPISODE_IDENTIFIER,ADMISSION_DATE,ADMISSION_TIME,AGE_AT_ADMISSION,DISCHARGE_DATE,MAIN_SPECIALTY_CODE,MAIN_SPECIALTY_CODE_DESC,...,time_to_culture_hours,esbl_status,BIRTH_DATE,DEATH_DATE,ETHNICITY,ETHNICITY_DES,GENDER,GENDER_DESC,UPDATE_DT_TM,esbl_label_test
0,10001,7867287,2998828,3749211,2023-02-12,00:52:00,64,2023-02-26,140,oral surgery,...,NaN,unknown,1958-08-07,2021-01-03,L,Any other Asian background,M,Male,2023-01-05,0
1,10002,6551735,8909338,9353593,2022-11-16,23:43:00,72,2022-12-01,320,cardiology,...,216.0,unknown,1955-02-09,2023-11-15,N,Black or Black British - African,U,Unknown,2023-07-03,0
2,10003,4441228,1307783,3518105,2022-05-12,20:10:00,29,2022-05-27,320,cardiology,...,292.0,unknown,1963-03-27,2025-10-17,A,White - British,U,Unknown,2023-10-23,1
3,10004,5593202,5695692,6593442,2023-10-11,00:52:00,19,2023-10-30,300,general medicine,...,NaN,unknown,1976-04-02,2025-10-28,M,Black or Black British - Caribbean,I,Indeterminate,2023-10-10,0
4,10005,6532182,7455884,6203572,2023-08-14,05:42:00,50,2023-08-18,300,general medicine,...,NaN,unknown,1986-04-04,2024-10-18,K,Asian or Asian British - Bangladeshi,I,Indeterminate,2023-01-06,0


In [5]:
print(
    "\n" + "!" * 80 +
    "\nWARNING: esbl_label_test IS BEING MANUALLY GENERATED FOR DEMOS ONLY!"
    "\nThese values are randomly generated and must NOT be used for real analysis."
    "\n" + "!" * 80 + "\n"
)

df["esbl_label_test"] = np.random.choice(
    [0, 1],
    size=len(df),
    p=[0.8, 0.2]
)

df.to_parquet(input_cfg['path'])


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
These values are randomly generated and must NOT be used for real analysis.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!



In [6]:
required = config['validation']['required_columns']
structure_validation = validate_admission_table(
    df,
    required_columns=required,
    admission_id=columns['admission_id'],
    date_columns=[
        columns['admission_date'],
        columns['discharge_date']
    ],
)
target_validation = validate_binary_target(df, columns['target'])

print('Structural validation')
display(structure_validation)
print('\nTarget validation')
display(target_validation)
print('\nObserved target values')
display(df[columns['target']] \
        .value_counts(dropna=False) \
        .rename_axis('value') \
        .reset_index(name='n'))

Structural validation


{'n_rows': 10000,
 'n_columns': 189,
 'is_empty': False,
 'admission_id_present': True,
 'duplicate_admission_rows': 20,
 'date_parse_failures': {'ADMISSION_DATE': 0, 'DISCHARGE_DATE': 0},
 'warnings': ['Duplicate values found in SPELL_IDENTIFIER']}


Target validation


{'target_column': 'esbl_label_test',
 'n_missing': 0,
 'unique_values': [0, 1],
 'n_unique_non_missing': 2,
 'is_binary': True}


Observed target values


,value,n
0,0,7974
1,1,2026


In [7]:
report = {
    'dataset_overview': dataset_overview(df),
    'column_summary': column_summary(df),
    'duplicate_summary': duplicate_summary(df, [columns['admission_id'], columns['patient_id']]),
    'outcome_summary': outcome_summary(df, columns['target']),
    'temporal_summary': temporal_summary(
        df,
        date_column=columns['admission_date'],
        target_column=columns['target'],
        freq=config['analysis']['temporal_frequency'],
        #target_mapping=target_mapping,
    ),
}
for name, table in report.items():
    print(f'\n--- {name} ---')
    display(table.head(20))

save_audit_report(report, config['outputs']['audit_directory'])
print(f"Saved audit tables to {config['outputs']['audit_directory']}")


--- dataset_overview ---


,n_rows,n_columns,n_subjects,n_admissions,memory_mb
0,10000,189,10000,9990,21.83



--- column_summary ---


,column,dtype,n_rows,n_missing,missing_pct,n_unique,example_values
175,days_since_last_discharge,float64,10000,10000,100.00,0,
176,qualifying_last_abx,object,10000,10000,100.00,0,
115,charlson_hx_hemiplegia_dsfirst,float64,10000,9999,99.99,1,164.78680555555556
114,charlson_hx_hemiplegia_dslast,float64,10000,9999,99.99,1,164.78680555555556
136,charlson_hx_cancer_met_dsfirst,float64,10000,9995,99.95,5,"525.2465277777778, 944.11875, 1235.95069444444..."
135,charlson_hx_cancer_met_dslast,float64,10000,9995,99.95,5,"525.2465277777778, 944.11875, 1235.95069444444..."
124,charlson_hx_cancer_solid_dsfirst,float64,10000,9995,99.95,5,"525.2465277777778, 944.11875, 1235.95069444444..."
123,charlson_hx_cancer_solid_dslast,float64,10000,9995,99.95,5,"525.2465277777778, 944.11875, 1235.95069444444..."
127,charlson_hx_leukemia_dsfirst,float64,10000,9994,99.94,6,"676.3152777777777, 292.13194444444446, 220.885..."
126,charlson_hx_leukemia_dslast,float64,10000,9994,99.94,6,"676.3152777777777, 292.13194444444446, 220.885..."



--- duplicate_summary ---


,identifier,n_non_missing,n_unique,n_rows_in_duplicate_groups,has_duplicates
0,SPELL_IDENTIFIER,10000,9990,20,True
1,SUBJECT,10000,10000,0,False



--- outcome_summary ---


,value,n,percentage
0,0,7974,79.74
1,1,2026,20.26



--- temporal_summary ---


,period,n_admissions,n_events,n_missing_outcome,event_rate
0,2020-01-01,2486,560.0,0,0.225261
1,2021-01-01,2513,478.0,0,0.190211
2,2022-01-01,2472,504.0,0,0.203883
3,2023-01-01,2527,483.0,0,0.191136
4,2024-01-01,2,1.0,0,0.5


Saved audit tables to results/audit
